<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/ml/pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model on a billiards dataset and exports **TFLite FP16**
ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU — Runtime → Change runtime type → T4 GPU.

The dataset may include multiple classes (balls, table, pockets, etc.).
The model is trained on all classes as-is. You only need to know which
class index corresponds to pockets — set that in Section 2 and the
smoke-test and Android integration will use it correctly.

In [2]:
from ultralytics import YOLO
import os

EPOCHS = 50
checkpoint_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'last.pt')

if os.path.exists(checkpoint_path):
    model = YOLO(checkpoint_path)
    print(f'Resuming from checkpoint: {checkpoint_path}')
else:
    model = YOLO('yolov8n.pt')
    print('No checkpoint found. Starting fresh training with yolov8n.pt')

model.train(
    data=os.path.join(COMBINED_DIR, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name='pocket_detector',
    resume=os.path.exists(checkpoint_path)
)

ModuleNotFoundError: No module named 'ultralytics'

In [15]:
import os
import shutil
import yaml

def merge_datasets(source_paths, target_path):
    master_classes = []
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(target_path, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(target_path, split, 'labels'), exist_ok=True)

    for folder in source_paths:
        y_path = os.path.join(folder, 'data.yaml')
        if not os.path.exists(y_path):
            print(f'Warning: data.yaml not found in {folder}')
            continue

        with open(y_path, 'r') as f:
            cfg = yaml.safe_load(f)
            names = cfg.get('names', [])
            if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
            class_map = {i: (master_classes.index(n) if n in master_classes else (master_classes.append(n) or len(master_classes)-1)) for i, n in enumerate(names)}

        for split in ['train', 'valid', 'test']:
            src_split = split if os.path.exists(os.path.join(folder, split)) else 'val' if split == 'valid' else split
            s_img, s_lbl = os.path.join(folder, src_split, 'images'), os.path.join(folder, src_split, 'labels')
            if os.path.exists(s_img):
                for f in os.listdir(s_img): shutil.copy2(os.path.join(s_img, f), os.path.join(target_path, split, 'images', f))
            if os.path.exists(s_lbl):
                for f in os.listdir(s_lbl):
                    with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
                    with open(os.path.join(target_path, split, 'labels', f), 'a') as lf:
                        for l in lines:
                            parts = l.split()
                            if parts: parts[0] = str(class_map[int(parts[0])]); lf.write(' '.join(parts) + '\n')

    with open(os.path.join(target_path, 'data.yaml'), 'w') as f:
        yaml.dump({'train': f'{target_path}/train/images', 'val': f'{target_path}/valid/images', 'test': f'{target_path}/test/images', 'nc': len(master_classes), 'names': master_classes}, f)
    return master_classes

# Point these to your ALREADY EXTRACTED folder paths
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/Pool-Table-Detection-6',
    '/kaggle/input/cue-detat'
]

master_classes = merge_datasets(sources, COMBINED_DIR)
print(f'Merged Classes: {master_classes}')

KeyboardInterrupt: 

In [ ]:
import os
import yaml

dataset_yaml = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml'

if os.path.exists(dataset_yaml):
    with open(dataset_yaml, 'r') as f:
        data = yaml.safe_load(f)

    print('--- Dataset Configuration ---')
    for key, value in data.items():
        if key not in ['train', 'val', 'test']:
            print(f'{key}: {value}')

    print('\n--- Split Statistics ---')
    for split in ['train', 'val', 'test']:
        # Some Yaml use 'val', directory is usually 'valid' or 'val'
        path_key = split if split in data else 'val' if split == 'valid' else None
        if path_key and path_key in data:
            img_path = data[path_key]
            if os.path.exists(img_path):
                count = len([f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                print(f'{split.capitalize()} images: {count}')
            else:
                print(f'{split.capitalize()} path not found: {img_path}')
else:
    print(f'Error: {dataset_yaml} not found. Please ensure the merge step completed successfully.')

## 1. Install

In [12]:
!pip install -q ultralytics roboflow kagglehub
import ultralytics
ultralytics.checks()

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.3/112.6 GB disk)


In [14]:
from roboflow import Roboflow
import kagglehub

# 1. Download Dataset A (Roboflow)
rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("leonardo-wijaya-bdcih").project("pool-table-detection")
dataset_a = project.version(6).download("yolov8")

# 2. Download Dataset B (Kaggle)
kaggle_path = kagglehub.dataset_download("hereliesaz/cue-detat")

print(f"Data sources ready at: {dataset_a.location} and {kaggle_path}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pool-Table-Detection-6 in yolov8:: 100%|██████████| 6480/6480 [01:08<00:00, 94.81it/s] 


Using Colab cache for faster access to the 'cue-detat' dataset.
Data sources ready at: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/datasets/balls_dataset/Pool-Table-Detection-6 and /kaggle/input/cue-detat


In [ ]:
import zipfile
import os
import yaml

# List of zip files to extract
datasets_zips = [
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/Pool Table Detection.v6i.yolov8.zip'
]

def extract_and_inspect(zip_path):
    folder_name = os.path.basename(zip_path).replace('.zip', '').replace(' ', '_')
    extract_to = f'/content/{folder_name}'

    print(f'--- Processing: {os.path.basename(zip_path)} ---')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        yaml_path = os.path.join(extract_to, 'data.yaml')
        if os.path.exists(yaml_path):
            with open(yaml_path, 'r') as f:
                data = yaml.safe_load(f)
                names = data.get('names', [])
                print(f'Location: {extract_to}')
                print(f'Classes: {names}')
        return extract_to
    except Exception as e:
        print(f'Error extracting {zip_path}: {e}')
        return None

# Extract zip files and keep track of folders
extracted_paths = []
for ds in datasets_zips:
    if os.path.exists(ds):
        path = extract_and_inspect(ds)
        if path: extracted_paths.append(path)
    else:
        print(f'File not found: {ds}')

# Add the Roboflow downloaded folder to the list
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f'Added Roboflow dataset: {roboflow_dir}')

In [6]:
import glob
import os

# Search for any zip files in the content directory
found_zips = glob.glob('/content/*.zip')
print(f'Found zip files: {found_zips}')

# Filter for relevant datasets and update the list for cell 5761cd6d
datasets_zips = [f for f in found_zips if 'Billiards' in f or 'Pool' in f]

if not datasets_zips:
    print('No matching zip files found. Please upload your dataset zips to the /content folder.')
else:
    print(f'Updated dataset list to: {datasets_zips}')

# Re-run the inspection/extraction logic if paths are found
for ds in datasets_zips:
    path = extract_and_inspect(ds)
    if path and path not in extracted_paths:
        extracted_paths.append(path)

Found zip files: ['/content/Pool Table Detection.v6i.yolov8.zip', '/content/CueTor Billiards Training.v8i.yolov8.zip', '/content/pool-balls-yolov11m.v14i.yolov8.zip']
Updated dataset list to: ['/content/Pool Table Detection.v6i.yolov8.zip', '/content/CueTor Billiards Training.v8i.yolov8.zip']
--- Processing: Pool Table Detection.v6i.yolov8.zip ---
Error extracting /content/Pool Table Detection.v6i.yolov8.zip: File is not a zip file
--- Processing: CueTor Billiards Training.v8i.yolov8.zip ---
Location: /content/CueTor_Billiards_Training.v8i.yolov8
Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']


In [13]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Define core paths
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install -q roboflow
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

# Target path on Drive
balls_path = '/content/drive/MyDrive/billiards_training/datasets/balls_dataset'
os.makedirs(balls_path, exist_ok=True)
%cd {balls_path}

rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("wrckmn1").project("pool-balls-yolov11m")
version = project.version(14)
dataset = version.download("yolov8")

print(f"Balls dataset downloaded to Drive: {balls_path}")

/content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/datasets/balls_dataset
loading Roboflow workspace...
loading Roboflow project...
Balls dataset downloaded to Drive: /content/drive/MyDrive/billiards_training/datasets/balls_dataset


In [9]:
import os

# Manually re-populate extracted_paths based on existing directories
extracted_paths = []

# Check for the Roboflow folder
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f"Found Roboflow dataset: {roboflow_dir}")

# Check for any other folders created by extraction (excluding standard Colab folders)
for item in os.listdir('/content'):
    full_path = os.path.join('/content', item)
    if os.path.isdir(full_path) and item not in ['.config', 'drive', 'sample_data', 'combined_dataset']:
        if full_path not in extracted_paths:
            extracted_paths.append(full_path)
            print(f"Found extracted dataset: {full_path}")

print(f"Final list of datasets to merge: {extracted_paths}")

Found extracted dataset: /content/CueTor_Billiards_Training.v8i.yolov8
Final list of datasets to merge: ['/content/CueTor_Billiards_Training.v8i.yolov8']


In [10]:
import kagglehub
path = kagglehub.dataset_download("hereliesaz/cue-detat")

Using Colab cache for faster access to the 'cue-detat' dataset.


In [11]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

NameError: name 'master_classes' is not defined

In [ ]:
import os

# Verification Check
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/Pool-Table-Detection-6',
    '/kaggle/input/cue-detat'
]

print('--- Pre-Merge Validation ---')
for path in sources:
    if os.path.exists(path):
        y_path = os.path.join(path, 'data.yaml')
        has_yaml = os.path.exists(y_path)
        print(f'✅ Found: {path}')
        print(f'   - data.yaml: {"Found" if has_yaml else "MISSING"}')
    else:
        print(f'❌ NOT FOUND: {path}')

# Check if combined directory is ready
print(f'\nTarget directory for merging: {COMBINED_DIR}')

In [ ]:
from ultralytics import YOLO
import os

# Training Configuration
EPOCHS = 50
checkpoint_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'last.pt')

if os.path.exists(checkpoint_path):
    model = YOLO(checkpoint_path)
    print(f'Resuming from checkpoint: {checkpoint_path}')
else:
    model = YOLO('yolov8n.pt')
    print('No checkpoint found. Starting fresh training with yolov8n.pt')

# To make this 'pausable', we ensure save=True and set a save_period.
# You can stop the cell execution at any time and re-run this cell to resume.
model.train(
    data=os.path.join(COMBINED_DIR, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name='pocket_detector',
    save=True,
    save_period=5, # Saves a checkpoint every 5 epochs
    resume=os.path.exists(checkpoint_path)
)

In [ ]:
import glob
import os

# Search for any 'best.pt' files in the training directory
search_path = "/content/drive/MyDrive/billiards_training/**/best.pt"
found_weights = glob.glob(search_path, recursive=True)

if found_weights:
    print("Found weights at:")
    for w in sorted(found_weights):
        print(f"- {w}")

    # Pick the most recent one (usually the one with the highest number or latest timestamp)
    latest_weights = sorted(found_weights)[-1]
    print(f"\nRecommended weights for export: {latest_weights}")
else:
    print("No 'best.pt' files found. Please ensure training completed successfully.")

In [ ]:
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

    # Target path on Drive
    cuetor_path = '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8'
    os.makedirs(cuetor_path, exist_ok=True)
    %cd {cuetor_path}

    # Use Universe API approach for public datasets
    rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
    project = rf.universe("cuetor").project("cuetor-billiards-training")
    version = project.version(8)
    dataset = version.download("yolov8")

    print(f"CueTor dataset downloaded to Drive: {cuetor_path}")

In [ ]:
import os
import shutil
import yaml
import kagglehub

# 1. Download Kaggle dataset
kaggle_path = kagglehub.dataset_download("hereliesaz/cue-detat")

# 2. Source directories
extracted_paths = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/pool-balls-yolov11m-14',
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    kaggle_path
]

combined_dir = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset'
if os.path.exists(combined_dir): shutil.rmtree(combined_dir)
os.makedirs(combined_dir, exist_ok=True)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(combined_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(combined_dir, split, 'labels'), exist_ok=True)

master_classes = []

print("Merging ALL datasets (Drive + Kaggle)...")
for folder in extracted_paths:
    if not os.path.exists(folder):
        print(f"Warning: {folder} not found.")
        continue

    y_path = os.path.join(folder, 'data.yaml')
    if not os.path.exists(y_path):
        subdirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
        for sd in subdirs:
            if os.path.exists(os.path.join(folder, sd, 'data.yaml')):
                folder = os.path.join(folder, sd)
                y_path = os.path.join(folder, 'data.yaml')
                break

    if not os.path.exists(y_path): continue

    with open(y_path, 'r') as f:
        cfg = yaml.safe_load(f)
        names = cfg.get('names', [])
        if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]

        class_map = {}
        for i, name in enumerate(names):
            if name not in master_classes: master_classes.append(name)
            class_map[i] = master_classes.index(name)

    for split in ['train', 'valid', 'test']:
        src_split = split if os.path.exists(os.path.join(folder, split)) else 'val' if split == 'valid' else split
        s_img, s_lbl = os.path.join(folder, src_split, 'images'), os.path.join(folder, src_split, 'labels')

        if os.path.exists(s_img):
            for f in os.listdir(s_img):
                shutil.copy2(os.path.join(s_img, f), os.path.join(combined_dir, split, 'images', f))
        if os.path.exists(s_lbl):
            for f in os.listdir(s_lbl):
                with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
                with open(os.path.join(combined_dir, split, 'labels', f), 'w') as lf:
                    for l in lines:
                        p = l.split()
                        if p:
                            p[0] = str(class_map.get(int(p[0]), p[0]))
                            lf.write(' '.join(p) + '\n')

with open(os.path.join(combined_dir, 'data.yaml'), 'w') as f:
    yaml.dump({'train': f'{combined_dir}/train/images', 'val': f'{combined_dir}/valid/images', 'test': f'{combined_dir}/test/images', 'nc': len(master_classes), 'names': master_classes}, f)

print(f"All datasets combined on Drive: {combined_dir}")
print(f"Final Classes: {master_classes}")

In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

## 2. Configuration

Fill these in, then run all cells.

## 3. Load Dataset & Inspect Classes

## 4. Train

In [ ]:
from ultralytics import YOLO
import os
import torch

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 16 # Adjusted for T4 RAM stability
DATASET_YAML = "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Find the index for 'pool-table-hole' dynamically
try:
    pocket_idx = master_classes.index('pool-table-hole')
    POCKET_CLASS_IDX = [pocket_idx]
    print(f"Detected pocket class index: {POCKET_CLASS_IDX}")
except (NameError, ValueError):
    POCKET_CLASS_IDX = [1]
    print("Defaulting POCKET_CLASS_IDX to [1]")

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    name="pocket_detector",
    project=PROJECT_DIR,
    save=True
)

print(f"Training complete. Weights are in: {results.save_dir}")

In [ ]:
from ultralytics import YOLO
import os

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = "/content/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Ensure the base directory exists before training starts
os.makedirs(PROJECT_DIR, exist_ok=True)

# Resume Logic: Check if 'last.pt' exists in the weights folder
checkpoint_path = os.path.join(PROJECT_DIR, "pocket_detector", "weights", "last.pt")
resume_training = os.path.exists(checkpoint_path)

if resume_training:
    print(f"Found checkpoint at {checkpoint_path}. Resuming training...")
    model = YOLO(checkpoint_path)
else:
    print("No checkpoint found. Starting fresh training with yolov8n.pt")
    model = YOLO("yolov8n.pt")

# Train/Resume the model
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project=PROJECT_DIR,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    save=True,
    save_period=5,
    resume=resume_training
)

print(f"Training complete or paused. Weights are in: {results.save_dir}")

In [ ]:
import os

# --- GitHub Configuration ---
# IMPORTANT: Fill these in with your actual details!
GITHUB_TOKEN = "your_token_here"
GITHUB_USER = "your_username"
GITHUB_REPO = "your_repo_name"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Navigate to your drive project folder
if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}

    # Initialize git if not already done
    if not os.path.exists('.git'):
        !git init
        !git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git
        !git branch -M main

    # Configure user
    !git config --global user.email "you@example.com"
    !git config --global user.name "{GITHUB_USER}"

    # Add, commit and push
    !git add .
    !git commit -m "Update training weights and logs"
    !git push -u origin main
    print("Changes pushed to GitHub.")
else:
    print(f"Error: {PROJECT_DIR} not found. Ensure training has started and created this directory.")

## 5. Validate

In [ ]:
import os
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
if metrics.box.map50 > 0.85:
    print("✓ mAP50 > 0.85 — good to export")
else:
    print("⚠ mAP50 below 0.85 — consider more epochs or more data")

## 6. Export to TFLite FP16

In [ ]:
from ultralytics import YOLO
import os
import shutil

# Define paths
weights_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'best.pt')
drive_export_dir = '/content/drive/MyDrive/billiards_training/exports'
os.makedirs(drive_export_dir, exist_ok=True)

# Load model
model = YOLO(weights_path)

# Export to TFLite FP16
print("Exporting to TFLite...")
export_result = model.export(
    format='tflite',
    imgsz=640,
    half=True,
    nms=True
)

# The export method returns the path to the exported file/folder
target_tflite = os.path.join(drive_export_dir, 'pocket_detector_fp16.tflite')

if export_result.endswith('.tflite'):
    shutil.copy2(export_result, target_tflite)
else:
    # If it exported to a directory, find the .tflite file inside
    tflite_files = [f for f in os.listdir(export_result) if f.endswith('.tflite')]
    if tflite_files:
        shutil.copy2(os.path.join(export_result, tflite_files[0]), target_tflite)

print(f'Export complete! TFLite file saved to Drive: {target_tflite}')

## 7. Smoke-test

Shows detections for the pocket class only (filtered by `POCKET_CLASS_IDX`).

In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()
SZ    = int(inp_d[0]['shape'][1])
dtype = inp_d[0]['dtype']
print("Input:", inp_d[0]['shape'], dtype)

def infer(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], w, h

yaml_dir = os.path.dirname(DATASET_YAML) + "/"
imgs = (glob.glob(yaml_dir + 'valid/images/*.jpg') +
        glob.glob(yaml_dir + 'valid/images/*.png'))[:4]

fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(5 * max(len(imgs), 1), 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        # d = [x1, y1, x2, y2, conf, class_idx]  (with baked NMS)
        conf = float(d[4]) if len(d) >= 5 else 0
        cls  = int(d[5])   if len(d) >= 6 else 0
        if conf >= CONF_THRESHOLD and cls in POCKET_CLASS_IDX:
            x1,y1,x2,y2 = (int(d[0]*w/SZ), int(d[1]*h/SZ),
                            int(d[2]*w/SZ), int(d[3]*h/SZ))
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(draw, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')

plt.tight_layout(); plt.show()

## 8. Download

In [ ]:
import shutil
import os

# Verify weights directory on Drive
train_run_weights = '/content/drive/MyDrive/billiards_training/pocket_detector/weights'
backup_dir = '/content/drive/MyDrive/billiards_training/snapshots_backup'

if os.path.exists(train_run_weights):
    if os.path.exists(backup_dir): shutil.rmtree(backup_dir)
    shutil.copytree(train_run_weights, backup_dir)
    print(f'All snapshots (best.pt, last.pt, etc.) backed up to: {backup_dir}')
else:
    print(f'Warning: Weights directory not found at {train_run_weights}. Check PROJECT_DIR.')

## 9. Android Integration (v1.4)

1. Copy `pocket_detector.tflite` → `app/src/main/assets/`

2. `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.16.1")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
```

3. Implement `PocketDetector` — filter output by the pocket class index from this notebook:
```kotlin
class TflitePocketDetector(context: Context) : PocketDetector {
    // Class indices that correspond to pockets in your dataset
    private val pocketClassIndices = setOf(0)  // match POCKET_CLASS_IDX above

    private val interpreter = Interpreter(
        FileUtil.loadMappedFile(context, "pocket_detector.tflite")
    )

    override fun detect(yBytes: ByteArray, width: Int, height: Int): List<PointF>? {
        // 1. Resize yBytes to 640x640, replicate Y channel to RGB
        // 2. Run interpreter
        // 3. Parse output [x1,y1,x2,y2,conf,class_idx]
        //    return centres where conf >= 0.4 && class_idx in pocketClassIndices
    }
}
```

4. Pass to `TableScanAnalyzer` in `ProtractorScreen.kt`:
```kotlin
val detector = remember { TflitePocketDetector(context) }
val tableScanAnalyzer = remember(tableScanViewModel) {
    TableScanAnalyzer(
        tableScanViewModel::onFrame,
        tableScanViewModel::onFeltColorSampled,
        pocketDetector = detector,
    )
}
```

# Task
Clean and consolidate the billiards pocket detector training pipeline. Install `ultralytics`, `roboflow`, and `kagglehub`; mount Google Drive; and define `PROJECT_DIR` and `COMBINED_DIR`. Download the "cue-detat" dataset from kagglehub and save it to "/content/drive/MyDrive/billiards_training/datasets/". Perform a consolidated merge of the four datasets ("pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat") from their respective Drive/Kaggle paths into `COMBINED_DIR`, generating a master `data.yaml`. Dynamically identify the class index for 'pool-table-hole' as `POCKET_CLASS_IDX`. Execute a pausable YOLOv8n training session with checkpoints saved to Drive. Finally, validate the model, export it to TFLite FP16 format in the Drive exports folder, and remove all redundant cells from the notebook.

## Consolidate Setup

### Subtask:
Install required libraries, mount Google Drive, and define global path variables for the project and combined dataset.


**Reasoning**:
I will consolidate the setup by installing dependencies, mounting Google Drive, and defining the necessary project path variables as requested in the subtask instructions.



In [16]:
!pip install -q ultralytics roboflow kagglehub

import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define global path variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# Ensure directories exist
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

print(f'Project Directory: {PROJECT_DIR}')
print(f'Combined Dataset Directory: {COMBINED_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project Directory: /content/drive/MyDrive/billiards_training
Combined Dataset Directory: /content/drive/MyDrive/billiards_training/datasets/combined_dataset


## Download Cue d'Etat to Drive

### Subtask:
Download the 'cue-detat' dataset from Kaggle and move it to the persistent project directory on Google Drive.


**Reasoning**:
I will add a code block to download the 'cue-detat' dataset from Kaggle and transfer it to the persistent directory on Google Drive as instructed.



In [17]:
import kagglehub
import shutil
import os

# 1. Download the dataset from Kaggle
kaggle_src_path = kagglehub.dataset_download('hereliesaz/cue-detat')

# 2. Define the destination path on Google Drive
drive_dest_path = '/content/drive/MyDrive/billiards_training/datasets/cue-detat'

# 3. Move or copy the dataset to Google Drive
# Remove existing directory if it exists to avoid errors during copy
if os.path.exists(drive_dest_path):
    shutil.rmtree(drive_dest_path)

shutil.copytree(kaggle_src_path, drive_dest_path)

# 4. Verify the transfer
if os.path.exists(drive_dest_path):
    print(f'Dataset successfully moved to: {drive_dest_path}')
    print(f'Files in directory: {os.listdir(drive_dest_path)}')
else:
    print(f'Error: Failed to move dataset to {drive_dest_path}')

Using Colab cache for faster access to the 'cue-detat' dataset.


KeyboardInterrupt: 

# Task
None

# Task
Clean and consolidate the billiards pocket detector training pipeline. Start by installing `ultralytics`, `roboflow`, and `kagglehub`, mounting Google Drive, and defining project variables `PROJECT_DIR` and `COMBINED_DIR`.

The core task involves preparing the data by:
1. Extracting the reuploaded zip files in `/content` (specifically "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip") and persisting them to Google Drive.
2. Downloading the "cue-detat" dataset from `kagglehub` and moving it to "/content/drive/MyDrive/billiards_training/datasets/cue-detat".
3. Executing a consolidated merge of all four datasets ("pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat") from their persistent Drive paths into `COMBINED_DIR`, generating a master `data.yaml`.

Following data preparation:
1. Dynamically detect the class index for 'pool-table-hole' from the merged dataset and define it as `POCKET_CLASS_IDX`.
2. Configure and run a pausable YOLOv8n training session that supports resuming from Drive checkpoints and saves weights periodically.
3. Validate the best weights and export the model to TFLite FP16 format in the Drive exports folder.
4. Clean the notebook by removing redundant cells and reordering the functional cells into a logical pipeline.

## Extract and Persist Reuploaded Data

### Subtask:
Identify the reuploaded zip files in /content, extract their contents, and move them to the persistent datasets directory on Google Drive.


**Reasoning**:
I will create a code block to extract the three specific zip files from the /content directory and move their contents to the persistent project directory on Google Drive as instructed.



In [18]:
import zipfile
import shutil
import os

# 1. Define list of zip files in /content
dataset_zips = [
    '/content/Pool Table Detection.v6i.yolov8.zip',
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/pool-balls-yolov11m.v14i.yolov8.zip'
]

# Map zip filenames to a cleaner folder name for Drive
zip_to_folder = {
    'Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    'CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    'pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

print('--- Extracting and Persisting Datasets ---')

for zip_path in dataset_zips:
    if not os.path.exists(zip_path):
        print(f'File not found: {zip_path}')
        continue

    zip_name = os.path.basename(zip_path)
    folder_name = zip_to_folder[zip_name]

    # 2. Extract contents to a temporary directory in /content
    temp_extract_path = f'/content/temp_{folder_name}'
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_extract_path)

    # 3. Define target directory in DATASETS_BASE (Google Drive)
    drive_target_path = os.path.join(DATASETS_BASE, folder_name)

    # 4. Move extracted folder to persistent location on Google Drive
    if os.path.exists(drive_target_path):
        shutil.rmtree(drive_target_path)

    shutil.move(temp_extract_path, drive_target_path)

    # 5. Print final paths to verify
    print(f'✅ Extracted and moved {zip_name} to:')
    print(f'   {drive_target_path}')

--- Extracting and Persisting Datasets ---


OSError: [Errno 22] Invalid argument

# Task
Clean and consolidate the billiards pocket detector training pipeline by first installing `ultralytics`, `roboflow`, and `kagglehub`, mounting Google Drive, and defining project variables `PROJECT_DIR` and `COMBINED_DIR`.

The data preparation phase involves:
1. Extracting the three reuploaded zip files in `/content` (specifically "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip") using robust extraction logic and persisting the resulting folders to Google Drive at "/content/drive/MyDrive/billiards_training/datasets/".
2. Downloading the "cue-detat" dataset from `kagglehub` and moving it to the persistent Drive path "/content/drive/MyDrive/billiards_training/datasets/cue-detat".
3. Executing a consolidated four-way merge of "pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat" from their persistent Drive locations into `COMBINED_DIR`, generating a unified `data.yaml`.

Post-data preparation:
1. Dynamically identify the class index for 'pool-table-hole' from the merged master class list and define it as `POCKET_CLASS_IDX`.
2. Configure and execute a pausable YOLOv8n training session that supports resuming from Drive checkpoints (`last.pt`) and utilizes a `save_period` for periodic progress saving.
3. Validate the final model using the best weights and export it to TFLite FP16 format in the Drive exports folder.
4. Finalize the notebook by removing all redundant or failed cells and organizing the functional code into a streamlined end-to-end pipeline.

## Fix Extraction and Setup Paths

### Subtask:
Initialize project directories and implement robust logic to extract reuploaded zip files while handling potential path or OS errors.


**Reasoning**:
I will consolidate the directory setup and implement robust zip extraction logic with error handling as requested in the subtask instructions.



In [ ]:
import os
import shutil

# 1. Define core directory variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# 2. Ensure directories exist on Google Drive
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

# 3. Map zip filenames to clean target folder names
zip_mapping = {
    'Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    'CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    'pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

print('--- Starting Shell-based Extraction Process ---')

# 4. Loop to extract each zip file using shell unzip for robustness
for zip_name, target_folder in zip_mapping.items():
    zip_path = os.path.join('/content', zip_name)

    if not os.path.exists(zip_path):
        print(f'Skipping: {zip_name} (File not found in /content)')
        continue

    try:
        temp_extract_path = os.path.join('/content', f'temp_{target_folder}')
        if os.path.exists(temp_extract_path):
            shutil.rmtree(temp_extract_path)
        os.makedirs(temp_extract_path, exist_ok=True)

        print(f'Extracting {zip_name} via shell...')
        # Use !unzip via os.system for better compatibility with large files/drive mount
        exit_code = os.system(f'unzip -q "{zip_path}" -d "{temp_extract_path}"')

        if exit_code != 0:
            raise OSError(f"Unzip failed with exit code {exit_code}")

        # Move to persistent location on Google Drive
        drive_target_path = os.path.join(DATASETS_BASE, target_folder)
        if os.path.exists(drive_target_path):
            shutil.rmtree(drive_target_path)

        shutil.move(temp_extract_path, drive_target_path)
        print(f'✅ Successfully persisted to: {drive_target_path}')

    except Exception as e:
        print(f'❌ Error processing {zip_name}: {e}')

print('--- Extraction Process Complete ---')

--- Starting Shell-based Extraction Process ---
Extracting Pool Table Detection.v6i.yolov8.zip via shell...
❌ Error processing Pool Table Detection.v6i.yolov8.zip: Unzip failed with exit code 3072
Extracting CueTor Billiards Training.v8i.yolov8.zip via shell...
✅ Successfully persisted to: /content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8
Extracting pool-balls-yolov11m.v14i.yolov8.zip via shell...
